# CardioIA — análise do baseline tabular

Este notebook reproduz o treinamento, avalia o modelo no conjunto de teste e apresenta incerteza, desempenho por sexo, calibração e interpretação das variáveis. A análise utiliza somente a modalidade numérica Cleveland e tem finalidade acadêmica.

## Preparação e execução

O pré-processamento é ajustado apenas no treino dentro de um `Pipeline`. A divisão é estratificada, com 80% para treino e 20% para teste e semente 42.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

RAIZ = Path.cwd()
if not (RAIZ / 'fase2').exists():
    if RAIZ.name == 'notebooks' and RAIZ.parent.name == 'fase2':
        RAIZ = RAIZ.parents[1]
    else:
        raise RuntimeError('Abra o notebook a partir da raiz do repositório ou da pasta fase2/notebooks.')
os.chdir(RAIZ)

subprocess.run([sys.executable, 'fase2/src/treinar_baselines.py'], check=True)
subprocess.run([sys.executable, 'fase2/src/analisar_modelo.py'], check=True)

## Comparação dos modelos

In [ ]:
import pandas as pd
from IPython.display import display

cv = pd.read_csv('fase2/resultados/metricas_validacao_cruzada.csv')
display(cv.pivot(index='modelo', columns='metrica', values='media').round(3))

teste = pd.read_csv('fase2/resultados/metricas_teste.csv')
display(teste.round(3))

## Incerteza e desempenho por sexo

Os intervalos de confiança de 95% são estimados por 2.000 reamostragens bootstrap. Os resultados dos subgrupos devem ser lidos com cautela porque o teste contém poucos pacientes, principalmente mulheres.

In [ ]:
subgrupos = pd.read_csv('fase2/resultados/metricas_por_sexo.csv')
intervalos = pd.read_csv('fase2/resultados/intervalos_confianca_bootstrap.csv')
display(subgrupos.round(3))
display(intervalos.round(3))

## Visualizações de desempenho

In [ ]:
from IPython.display import Image, display

graficos = [
    'matriz_confusao.png',
    'curva_roc.png',
    'curva_precisao_recall.png',
    'curva_calibracao.png',
    'distribuicao_alvo_por_sexo.png',
]
for grafico in graficos:
    display(Image(filename=f'fase2/resultados/graficos/{grafico}', width=720))

## Interpretação das variáveis

Coeficientes positivos aumentam o logaritmo da chance predita da classe `presente`; coeficientes negativos reduzem essa chance. Eles representam associação no modelo, não causalidade clínica.

In [ ]:
importancias = pd.read_csv('fase2/resultados/importancia_variaveis.csv')
display(importancias.head(20).round(4))
display(Image(filename='fase2/resultados/graficos/importancia_variaveis.png', width=850))

## Limitações

- A base contém somente 303 pacientes de uma instituição dos Estados Unidos, coletados na década de 1980.
- O conjunto de teste possui apenas 61 pacientes; métricas gerais e por sexo têm incerteza relevante.
- A avaliação por sexo é descritiva e não demonstra, sozinha, presença ou ausência de viés.
- O desempenho não pode ser generalizado para a população brasileira nem utilizado em decisões clínicas.
- Dados numéricos, textuais e visuais permanecem como modalidades independentes.